# Tech Challenge A - Fase 1

Projeto de classificação de câncer de mama com foco em apoio ao diagnóstico para saúde da mulher. O notebook cobre carga dos dados, Análise Exploratória de Dados, pré-processamento, treino, avaliação, explicabilidade e discussão crítica.

In [ ]:
from pathlib import Path
import warnings

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

OUTPUT_DIR = Path('../outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Download e carga do dataset

O dataset é baixado via [`kagglehub`](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data/data). Em seguida, o notebook identifica automaticamente o arquivo CSV dentro do diretório baixado.

In [ ]:
path = kagglehub.dataset_download('uciml/breast-cancer-wisconsin-data')
dataset_path = Path(path)
csv_files = sorted(dataset_path.rglob('*.csv'))
if not csv_files:
    raise FileNotFoundError(f'Nenhum CSV encontrado em {dataset_path}')
csv_path = csv_files[0]
df = pd.read_csv(csv_path)
print(f'Dataset path: {dataset_path}')
print(f'CSV selecionado: {csv_path.name}')
print(df.shape)

In [ ]:
df.head()

## 2. Entendimento inicial da base

A base contém atributos numéricos derivados de exames de massa mamária e a classe de diagnóstico `M` (maligno) ou `B` (benigno).

In [ ]:
df.info()
df.describe().T.head(10)

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0].head(10)

In [ ]:
if 'Unnamed: 32' in df.columns:
    df = df.drop(columns=['Unnamed: 32'])
if 'id' in df.columns:
    df = df.drop(columns=['id'])

df['rotulo_diagnostico'] = df['diagnosis'].map({'B': 'Benigno', 'M': 'Maligno'})
df['target'] = df['diagnosis'].map({'B': 0, 'M': 1})
df[['diagnosis', 'rotulo_diagnostico', 'target']].head()

## 3. Análise exploratória objetiva

Nesta etapa, o foco é entender o balanceamento da classe alvo, distribuições relevantes e correlações entre atributos que podem sugerir maior risco.

In [ ]:
class_counts = df['rotulo_diagnostico'].value_counts()
class_pct = df['rotulo_diagnostico'].value_counts(normalize=True).mul(100).round(2)
display(pd.DataFrame({'contagem': class_counts, 'percentual': class_pct}))

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x='rotulo_diagnostico', hue='rotulo_diagnostico', palette='Set2', legend=False)
ax.set_title('Distribuição do diagnóstico')
ax.set_xlabel('Classe')
ax.set_ylabel('Quantidade')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_distribuicao_diagnostico.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
top_features = ['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'concavity_mean']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for index, feature in enumerate(top_features):
    sns.boxplot(data=df, x='rotulo_diagnostico', y=feature, hue='rotulo_diagnostico', palette='Set2', ax=axes[index], legend=False)
    axes[index].set_title(feature)
    axes[index].set_xlabel('Diagnóstico')
    axes[index].set_ylabel('Valor')
    axes[index].set_xlabel('Diagnóstico')
    axes[index].set_ylabel('Valor')
for axis in axes[len(top_features):]:
    axis.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_boxplots_principais_variaveis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
corr = df.drop(columns=['diagnosis', 'rotulo_diagnostico']).corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Mapa de correlação das variáveis numéricas')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_heatmap_correlacao.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
corr_target = corr['target'].drop('target').sort_values(key=np.abs, ascending=False).head(10)
corr_target.to_frame('correlação_com_alvo')

### Interpretação da Análise Exploratória de Dados

Os atributos ligados a área, perímetro, raio e concavidade tendem a apresentar maior associação com a classe maligna. Isso sugere que alterações morfológicas mais intensas se conectam ao maior risco identificado pelo modelo.

## 4. Pré-processamento e separação treino/teste

A separação é feita com estratificação para preservar a proporção entre benigno e maligno nos conjuntos. A padronização é aplicada para beneficiar a Regressão Logística.

In [ ]:
feature_columns = [column for column in df.columns if column not in ['diagnosis', 'rotulo_diagnostico', 'target']]
X = df[feature_columns]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[('num', numeric_transformer, feature_columns)]
)

print('Treino:', X_train.shape, y_train.shape)
print('Teste:', X_test.shape, y_test.shape)

## 5. Treinamento dos modelos

Foram escolhidos dois modelos: Regressão Logística como baseline interpretável e Random Forest como modelo mais flexível para capturar relações não lineares.

In [ ]:
models = {
    'Regressão Logística': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(max_iter=2000, random_state=42))
    ]),
    'Random Forest': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', RandomForestClassifier(n_estimators=300, random_state=42))
    ])
}

fitted_models = {}
evaluation_rows = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    fitted_models[name] = model
    predictions[name] = y_pred
    evaluation_rows.append({
        'modelo': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred)
    })

results_df = pd.DataFrame(evaluation_rows).sort_values(by='recall', ascending=False)
results_df

## 6. Avaliação dos modelos

O recall da classe maligna é a métrica principal, porque um falso negativo em saúde pode atrasar encaminhamento e tratamento.

In [ ]:
styled_results = results_df.copy()
styled_results[['accuracy', 'precision', 'recall', 'f1_score']] = styled_results[['accuracy', 'precision', 'recall', 'f1_score']].round(4)
display(styled_results)

best_model_name = results_df.iloc[0]['modelo']
print(f'Modelo com melhor recall: {best_model_name}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for axis, (name, y_pred) in zip(axes, predictions.items()):
    cm_display = ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=axis, cmap='Blues', colorbar=False)
    cm_display.ax_.set_xlabel('Valor real')
    cm_display.ax_.set_ylabel('Valor predito')
    axis.set_title(f'Matriz de confusão - {name}')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_matrizes_confusao.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
for name, y_pred in predictions.items():
    print('=' * 80)
    print(name)
    print(classification_report(y_test, y_pred, target_names=['Benigno', 'Maligno']))

## 7. Explicabilidade

A explicabilidade combina importância das variáveis no Random Forest e SHAP no modelo vencedor. Para manter a execução enxuta, o SHAP é calculado sobre uma amostra pequena do conjunto de teste.

In [ ]:
rf_model = fitted_models['Random Forest']
rf_estimator = rf_model.named_steps['model']
rf_importances = pd.Series(rf_estimator.feature_importances_, index=feature_columns).sort_values(ascending=False).head(10)

plt.figure(figsize=(8, 5))
sns.barplot(x=rf_importances.values, y=rf_importances.index, palette='viridis')
plt.title('Top 10 variáveis mais importantes - Random Forest')
plt.xlabel('Importância')
plt.ylabel('Variável')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '05_feature_importance_random_forest.png', dpi=150, bbox_inches='tight')
plt.show()

rf_importances.to_frame('importância')

In [ ]:
best_model = fitted_models[best_model_name]
best_preprocessor = best_model.named_steps['preprocessor']
best_estimator = best_model.named_steps['model']

X_train_transformed = best_preprocessor.transform(X_train)
X_test_transformed = best_preprocessor.transform(X_test)
sample_size = min(50, X_test_transformed.shape[0])
X_shap = X_test_transformed[:sample_size]

explainer = shap.Explainer(best_estimator, X_train_transformed, feature_names=feature_columns)
shap_values = explainer(X_shap)
shap.plots.beeswarm(shap_values, max_display=10, show=False)
plt.xlabel('Impacto SHAP no resultado')
plt.ylabel('Variável')
cbar = plt.gcf().axes[-1]
cbar.set_ylabel('Valor da variável', rotation=270, labelpad=20)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '06_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Discussão

Os resultados indicam que os modelos conseguem diferenciar benigno e maligno com bom desempenho, mas isso não autoriza uso autônomo em ambiente clínico. A principal prioridade é reduzir falsos negativos, por isso o recall da classe maligna foi destacado. Ainda assim, o dataset é limitado, representa um contexto específico e não substitui validação clínica, governança de dados e interpretação médica. Na prática, o modelo pode apoiar a triagem e priorização de casos, mas a decisão final deve continuar com o profissional de saúde.

## 9. Conclusão

O notebook entrega um fluxo completo e reproduzível para classificação de risco em câncer de mama, atendendo aos requisitos principais da fase: análise exploratória, pré-processamento, comparação de modelos, métricas obrigatórias e explicabilidade.